1.LOAD DATASET

In [1]:
import pandas as pd
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
)

PLATFORM = "youtube"
SEEDS = [42, 123, 2024, 7, 99]
DATA_DIR = "../data_preprocess/processed_data/ml_data/"

results = []

for seed in SEEDS:
    train_df = pd.read_csv(f"{DATA_DIR}{PLATFORM}_train_seed{seed}.csv")
    test_df  = pd.read_csv(f"{DATA_DIR}{PLATFORM}_test_seed{seed}.csv")

    neg, pos = train_df["popularity"].value_counts()[0], train_df["popularity"].value_counts()[1]
    ratio = neg / pos

    X_train = train_df.drop(columns=["post_id", "user_id", "popularity"])
    y_train = train_df["popularity"]
    X_test  = test_df.drop(columns=["post_id", "user_id", "popularity"])
    y_test  = test_df["popularity"]

    model = XGBClassifier(
        objective="binary:logistic", eval_metric="logloss",
        scale_pos_weight=ratio, n_estimators=500, learning_rate=0.02,
        max_depth=6, min_child_weight=1, gamma=0.1,
        subsample=0.8, colsample_bytree=0.5,
        random_state=seed, n_jobs=-1
    )
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    results.append({
        "seed": seed,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred),
        "roc_auc": roc_auc_score(y_test, y_prob),
    })
    print(f"[seed {seed}] done | F1={results[-1]['f1']:.4f} | ROC-AUC={results[-1]['roc_auc']:.4f}")

results_df = pd.DataFrame(results)
print("\n" + "="*60)
print(f"{PLATFORM.upper()} BASELINE — MEAN ± STD ACROSS {len(SEEDS)} SEEDS")
print("="*60)
print(results_df.set_index("seed"))
print("\nMean ± Std:")
summary = results_df.drop(columns="seed").agg(["mean", "std"])
print(summary)

[seed 42] done | F1=0.6362 | ROC-AUC=0.8973
[seed 123] done | F1=0.6476 | ROC-AUC=0.9087
[seed 2024] done | F1=0.6685 | ROC-AUC=0.9154
[seed 7] done | F1=0.6484 | ROC-AUC=0.9050
[seed 99] done | F1=0.6365 | ROC-AUC=0.8994

YOUTUBE BASELINE — MEAN ± STD ACROSS 5 SEEDS
      accuracy  precision    recall        f1   roc_auc
seed                                                   
42    0.819912   0.533645  0.787586  0.636212  0.897344
123   0.823497   0.538955  0.811034  0.647577  0.908667
2024  0.835907   0.560748  0.827586  0.668524  0.915436
7     0.826806   0.545712  0.798621  0.648376  0.904992
99    0.820739   0.535278  0.784828  0.636465  0.899400

Mean ± Std:
      accuracy  precision    recall        f1   roc_auc
mean  0.825372   0.542867  0.801931  0.647431  0.905168
std   0.006478   0.011020  0.017680  0.013152  0.007283


In [2]:
results_df.insert(0, "model", "baseline_metadata")
results_df.insert(0, "platform", PLATFORM)

import os
os.makedirs("../results", exist_ok=True)
results_df.to_csv(f"../results/{PLATFORM}_baseline_metadata.csv", index=False)
print(f"Saved: ../results/{PLATFORM}_baseline_metadata.csv")

Saved: ../results/youtube_baseline_metadata.csv
